# Week 7: Web scraping with BeautifulSoup

This notebook follows [Part 11 of the course notes](https://craigfahner.github.io/INFO-664/part11/). We'll
look at what web scraping is, how to check whether a site can be scraped, how to read the structure of a
page, and how to turn the pages of a museum collection into a dataset saved as CSV and JSON.

The running example is the Cleveland Museum of Art's collection. Its `robots.txt` allows its artwork pages,
and its artwork pages are plain HTML.

**Ground rules for this notebook**

* We read `robots.txt` before we scrape anything.
* We identify ourselves honestly with a `User-Agent` header.
* We wait between requests (Cleveland asks for 10 seconds), and we **fetch each page only once**, saving it
  to a folder so that re-running the notebook doesn't send more requests.
* The only workaround we use for a site that doesn't allow scraping is **saving the page by hand** in your
  browser.
* Saved pages (the `cma_pages` folder that this notebook creates) stay **out of the GitHub repository**.

When you're ready, the two challenges build on this notebook:
[Challenge A: map the structure of a page](https://craigfahner.github.io/INFO-664/week7-challenge-a/) and
[Challenge B: extract the data and save it as JSON](https://craigfahner.github.io/INFO-664/week7-challenge-b/).

In [1]:
# If a library is missing, uncomment the next line, run it once, then restart the kernel.
# %pip install requests beautifulsoup4

import csv
import json
import os
import requests
from time import sleep
from bs4 import BeautifulSoup

## 1. What is web scraping?

An **API** (Part 10) hands us structured data on purpose. Many sites, including most museum collection pages,
have no API. What they have is **web pages**: HTML documents that a browser turns into headings, labels, and
images for a person to read.

**Web scraping** is writing a program to read those pages and pull out the pieces we want, turning a page meant
for people into structured data, like the lists and dictionaries from Part 3.

1. **Request** the page, with `requests`.
2. **Parse** the HTML into a structure Python can search.
3. **Find** the elements that hold the data we want.
4. **Extract** the text (or the link, or the image address) from them.
5. **Repeat** for more pages, and **save** the results.

## 2. What is BeautifulSoup?

HTML is just text, a long string full of tags. **BeautifulSoup** reads it and turns it into a tree of elements
that we can search by tag name, class, or position. HTML is built from the same pieces as the XML from Part 9:
**elements** inside opening and closing **tags**, which can hold text or other elements, and can have
**attributes** inside the opening tag.

Here is a simplified artwork page stored in a string:

In [2]:
html = """
<html>
  <body>
    <h1>Adeline Ravoux</h1>
    <p class="artist">Vincent van Gogh</p>
    <p class="date">1890</p>
    <a href="/art/1958.31">Artwork page</a>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")   # "html.parser" is the parser that comes with Python

`find()` returns the **first** element with a tag name. `.get_text()` returns just the text inside it.

In [3]:
title = soup.find("h1")
print(title)
print(title.get_text())

<h1>Adeline Ravoux</h1>
Adeline Ravoux


To find an element by its **class**, add `class_` (with an underscore, because `class` is a reserved word in
Python). `find_all()` returns a list of **every** match.

In [4]:
artist = soup.find("p", class_="artist")
print(artist.get_text())

for paragraph in soup.find_all("p"):
    print(paragraph.get_text())

Vincent van Gogh
Vincent van Gogh
1890


An element's attributes work like dictionary keys. Square brackets raise a `KeyError` if the attribute is missing, and `.get()` returns `None`.

In [5]:
link = soup.find("a")
print(link["href"])
print(link.get("href"))

/art/1958.31
/art/1958.31


## 3. Finding a website that can be scraped

Before choosing a site, work through these questions:

| Question | How to check |
| --- | --- |
| Is there an API or a data download? | Look in the menus and footer, or search for `<site> open data`. If there is one, use it. |
| Does the site allow automated access? | Read its `robots.txt` file. |
| Do the terms of use allow it? | Read the "Terms of Use" page, usually linked in the footer. |
| What is the content's license? | Open-access collections often say so on the page. |
| Is the data in the HTML? | **View Page Source** and search for a word from the data. If it isn't there, the page is built by JavaScript and `requests` will only get an empty shell. |
| Does it include personal information? | Be careful with anything about private individuals. |

### What is robots.txt?

Almost every site has a file at `/robots.txt` where its owners say which parts of the site automated programs
may visit, and how fast. **To read one, type `/robots.txt` after a site's main address in your browser.** Try it
now with the Cleveland Museum of Art: <https://www.clevelandart.org/robots.txt>. Here is what it said in
October 2026:

```text
# *
User-agent: *
Crawl-delay: 10
Disallow: /membership
Disallow: /orders
Disallow: /api
Disallow: /errors
Disallow: /404
Disallow: /500

# Host
Host: https://www.clevelandart.org

# Sitemaps
Sitemap: https://www.clevelandart.org/sitemap.xml
```

Lines starting with `#` are comments. The rest are **directives**:

| Directive | Meaning |
| --- | --- |
| `User-agent:` | Which robots the rules apply to. `*` means all of them. |
| `Disallow:` | A path that robots should **not** request. `Disallow: /membership` rules out any address that starts with `/membership`. |
| `Allow:` | An exception to a `Disallow`. |
| `Crawl-delay:` | How many seconds to wait between requests. |
| `Sitemap:` | The address of a list of the site's pages. |

Anything that isn't disallowed is allowed. Cleveland's file doesn't rule out its artwork pages (addresses
starting with `/art/`), but it asks for a 10-second delay. Compare that with the collection database of the
Milwaukee Art Museum, whose whole file is:

```text
User-agent: *
Disallow: /
```

`Disallow: /` blocks every path on the site, so that site asks robots not to visit it at all.

**How to read one.** Open the file in your browser, and then:

1. **If you get a "not found" page,** the site hasn't published any rules, so check its terms of use instead.
2. **Find the group that covers you.** `User-agent: *` means all robots, and that's the group that applies to a
   script like ours.
3. **Compare your page's address with each `Disallow:` line.** Take the part of the address after the domain,
   such as `/art/1958.31`. If it *starts with* the path on a `Disallow:` line, that page is off limits.
   `/membership/join` is blocked by `Disallow: /membership`, but `/art/1958.31` doesn't start with any of
   Cleveland's `Disallow:` paths, so it's allowed. A `Disallow:` with nothing after it blocks nothing.
4. **Look for an `Allow:` line**, which makes an exception to a `Disallow:`.
5. **Note the `Crawl-delay:`**, if there is one. That's how many seconds to wait between requests.
6. **Write down what you found,** including the date, since the rules can change.

`robots.txt` isn't a lock: nothing stops a program from ignoring it. It's a published request from the site's
owners, and respecting it is the expected behavior.

### Identifying yourself, and going slowly

* **Identify yourself.** By default, `requests` announces itself as `python-requests`, and some sites refuse
  that. We send our own `User-Agent` that says honestly who and what we are. This is about being upfront, not
  about disguise.
* **Wait between requests**, and respect any `Crawl-delay`.
* **Fetch a page once, save it, and experiment on the saved copy.** Working out which element holds what takes
  many tries, and this way each try doesn't send another request to the museum.

The function below does all three. It returns a page's HTML, fetching it only if we haven't already saved it
in the `cma_pages` folder. It uses `os.path.exists()` to check for the saved file:

In [6]:
headers = {"User-Agent": "INFO664-class-project (educational)"}

os.makedirs("cma_pages", exist_ok=True)


def fetch_page(url, filename):
    """Return a page's HTML, fetching it only if we haven't saved it yet."""
    if os.path.exists(filename):
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()

    response = requests.get(url, headers=headers)

    if not response.ok:
        print(url, "failed with status code", response.status_code)
        return None

    with open(filename, "w", encoding="utf-8") as file:
        file.write(response.text)

    sleep(10)    # Cleveland's robots.txt asks for 10 seconds between requests
    return response.text

## 4. What to do if robots.txt doesn't allow access

Sometimes the site you want says no. Here are the options, roughly in the order to try them:

1. **Look for an API or open data.** A site that blocks crawlers often publishes the same data a better way.
2. **Ask.** A short email to the museum's digital or collections team sometimes gets you a data file or
   permission in writing.
3. **Choose a different site.**
4. **Save the pages by hand.**

**In this course, saving pages by hand is the only workaround we use, and only for practice in class.**
`robots.txt` is aimed at *automated* requests. Opening a page in your own browser and saving it is ordinary
human use, and BeautifulSoup can then parse the saved file on your own computer with no request to the site.
Save a handful of pages that way, not hundreds.

What we **don't** do, even in class: disguise a program as a browser, get around "are you a robot?" pages or
login walls, use hidden search keys or internal addresses found in a site's code, or ignore a `Disallow` rule
because nothing technically stops us.

**How to save a page** (File → Save Page As, Cmd/Ctrl+S; these are Chrome's names):

* **"Webpage, HTML only"** saves exactly what the server sent, the same as `response.text`.
* **"Webpage, Complete"** saves the page as the browser has built it, including anything its JavaScript added.
  Use it for pages that fail the View Page Source test.
* **Developer tools:** Inspect → Elements → right-click the `<html>` element → Copy → Copy outerHTML, then paste
  it into a text editor and save it with a name ending in `.html`.

**Check the file** by opening it in a text editor and searching for a word from the data you want. If it's
missing, you saved the empty shell.

Everything after this step is the same no matter where the HTML came from. To parse a hand-saved file, open it
and pass its text to `BeautifulSoup()`.

## 5. Examining a single item: an artwork page

We'll work with Vincent van Gogh's portrait *Adeline Ravoux*:

`https://www.clevelandart.org/art/1958.31`

The end of the address, `1958.31`, is the work's **accession number**, the museum's own identifier. Every
artwork's page address is built from it.

We'd like to end up with a dictionary like this:

```python
{
    "title": "Adeline Ravoux",
    "artist": "Vincent van Gogh",
    "date": "1890",
    "culture": "Netherlands",
    "medium": "oil on fabric",
    "measurements": "Framed: 72.5 x 73.5 x 8.5 cm (...); Unframed: 50.2 x 50.5 cm (...)",
    "credit_line": "Bequest of Leonard C. Hanna Jr.",
    "accession_number": "1958.31",
    "location": "222 Impressionism & Post-Impressionism",
}
```

### Look at it in your browser first

Right-click the title and choose **Inspect**. The **Elements** panel shows the page's HTML as a tree. Hover
over elements to see what they highlight, click the triangles to open and close children, and search with
Cmd/Ctrl+F. Then run the View Page Source test: search the source for `oil on fabric`. It's there, so the data
is in the HTML that `requests` will see.

### Now with BeautifulSoup

This fetches the page once (about ten seconds, including the polite pause) and saves it:

In [7]:
url = "https://www.clevelandart.org/art/1958.31"

html = fetch_page(url, "cma_pages/cma_1958_31.html")
soup = BeautifulSoup(html, "html.parser")

print(len(html), "characters of HTML")

380677 characters of HTML


An HTML page has a `<head>`, which holds information *about* the page, and a `<body>`, which holds what you see. Start with the head:

In [8]:
print(soup.title.get_text())
print(soup.find("meta", property="og:title")["content"])

Adeline Ravoux | Cleveland Museum of Art
Adeline Ravoux | Cleveland Museum of Art


Now the first heading in the body:

In [10]:
print(soup.find("h1").get_text())

Artwork Page for Adeline Ravoux


That doesn't match anything on the screen. The `<h1>` has the class `sr-only`: it's for screen readers and
hidden from view. **What's in the HTML isn't always what's on the screen.**

### The details container

The facts are inside one element with an `id`, which is the easiest kind of element to find:

In [11]:
box = soup.find(id="artwork-details-container")

Elements nested inside another are its **children**. `find_all(True, recursive=False)` lists the *direct*
children (`True` means any tag):

In [16]:
children = box.find_all(True, recursive=False)

print(len(children))

for index, child in enumerate(children):
    print(index, child.name, "|", child.get_text(" ", strip=True)[:60])

14
0 h2 | Details / Information for Adeline Ravoux
1 div | Adeline Ravoux
2 div | 1890
3 div | Vincent van Gogh (Dutch, 1853–1890)
4 div | Culture Netherlands
5 div | Medium oil on fabric
6 div | Measurements Framed: 72.5 x 73.5 x 8.5 cm (28 9/16 x 28 15/1
7 div | Credit Line Bequest of Leonard C. Hanna Jr. 1958.31
8 div | Public Domain You can copy, modify, and distribute this work
9 div | Location 222 Impressionism & Post-Impressionism
10 div | ArtLens App Share Download Print
11 div | ? Did You Know? Adeline Ravoux, at age 13, was not pleased w
12 div | Description In May 1890, Vincent van Gogh arrived in Auvers,
13 div | Videos Artist Biography Provenance Citations Exhibition Hist


**How `enumerate()` works.** A `for` loop normally gives us one thing per pass: the next `child`. Here we also
wanted each child's *position* in the list, so that we could talk about "child 5". `enumerate()` takes a list
and hands back **pairs**: each position, followed by the item at that position. Here it is on a short list
(the `list()` is only there so we can see the pairs; `enumerate()` really makes them one at a time, as the loop
asks):

In [17]:
tags = ["h2", "div", "div"]

print(list(enumerate(tags)))

[(0, 'h2'), (1, 'div'), (2, 'div')]


The positions start at `0`, like list indexes. In our loop, `for index, child in enumerate(children):`
**unpacks** each pair into two variables. On the first pass `index` is `0` and `child` is the `<h2>`; on the
second, `index` is `1` and `child` is the first `<div>`; and so on to the end.

Without `enumerate()`, we'd have to keep the count ourselves:

In [18]:
index = 0

for child in children:
    print(index, child.name)
    index += 1

0 h2
1 div
2 div
3 div
4 div
5 div
6 div
7 div
8 div
9 div
10 div
11 div
12 div
13 div


Same numbers, but with extra bookkeeping, and an easy line to forget. (`enumerate(children, start=1)` would count
from `1` instead. We want `0` here because it matches the positions we use to pick a child out of the list, as
in `children[5]` below.)

Almost every fact we want is in that list:

* Children `1`, `2`, and `3` hold the title, the date, and the artist. They have no labels. We tell them apart by
  what they contain (the title is in an `<h3>`, the artist in an `<h4>`).
* Children `4` to `7`, and `9`, are **label and value rows**.

Look inside one of the rows:

In [22]:
row = children[5]

print(row)  # shows that there are two elements in children[5]

for part in row.find_all(True, recursive=False):
    print(part.name, "|", part.get_text(strip=True))

<div class="py-4 body-1-tight md:body-2-tight sm:border-solid sm:border-0 sm:border-t sm:border-gray-300 flex flex-col sm:grid sm:grid-cols-[10rem_1fr] sm:gap-x-4"><div class="font-bold mb-1 sm:mb-0">Medium</div><a class="underline underline-offset-2 first-letter:capitalize text-grey-05" href="/art/collection/search?medium=oil%20on%20fabric" target="_self">oil on fabric</a></div>
div | Medium
a | oil on fabric


Each row has a label in a `<div>` and a value after it (an `<a>`, a `<span>`, or another `<div>`). The
`Credit Line` row holds a link *and* a separate `<span>` with the accession number.

| Data | Where to find it |
| --- | --- |
| Title | The `<h3>` in child 1 (also in `<title>` and the `og:title` meta tag, with the museum name added) |
| Date | Child 2, a `<div>` with just the date |
| Artist | The `<a>` inside the `<h4>` in child 3. The nationality and years are in a `<span>` next to it |
| Culture, Medium, Measurements, Location | A row whose label `<div>` says `Culture`, `Medium`, and so on |
| Credit line | The `<a>` in the `Credit Line` row |
| Accession number | The `<span>` after the credit line link, and the end of the page's address |
| License | The `Public Domain` notice, child 8 |

### Which parts of the markup can we rely on?

| What to rely on | Why |
| --- | --- |
| An `id`, like `artwork-details-container` | It identifies one element and tends to stay put. |
| Tag names, like `h3` and `h4` | Headings are structural. |
| **Label text**, like `Medium` | People read it, so it changes rarely. Finding a row by its label survives other rows being added or removed. |
| **Position**, like "child 5" | Risky. A work with no `Culture` row would shift every row after it. |
| **Class names**, like `font-bold mb-1` | Avoid. They look like styling shorthand, not meaning, and can change at any time. |

### Optional: the JSON in the page

Near the bottom of the page's HTML is a `<script id="__NEXT_DATA__">` element holding a block of JSON that the
site uses to build the page. We won't use it: its structure belongs to the site's software, and many sites have
none. **We'll stay with the HTML tags.**

## 6. Scraping the artwork page

Now we pull each fact out of the HTML tags.

### The title, artist, and date

The title is the container's `<h3>`, and the artist is its `<h4>`. The `" "` in the artist's `get_text()` puts a
space between separate pieces of text in the element, which matters for works with an attribution like
*copy after Anthony van Dyck*.

In [23]:
title = box.find("h3").get_text(strip=True)
artist = box.find("h4").get_text(" ", strip=True)

print(title)
print(artist)

Adeline Ravoux
Vincent van Gogh


The date has no label and no tag of its own: it's a plain `<div>` right after the title. So we find it by its
**position in the tree**. An element's **parent** is the element that contains it, and **siblings** sit side by
side inside the same parent. We find the `<h3>`, go up to its parent `<div>`, then across to the next sibling
`<div>`:

In [24]:
title_block = box.find("h3").find_parent("div")
date = title_block.find_next_sibling("div").get_text(strip=True)

print(date)

1890


This is the most fragile lookup we'll write, since it depends on the date staying right after the title. When a page gives you labels, use them.

### The labeled rows

`find()` can search for an element by its text with `string=`. `box.find("div", string="Medium")` is "the
`<div>` whose text is exactly `Medium`", which is the label. The value is the label's next sibling:

In [25]:
label = box.find("div", string="Medium")
value = label.find_next_sibling()

print(value.get_text(" ", strip=True))

oil on fabric


We'll do this several times, so we write functions. They also check for `None`: a work might not *have* a given
row, and calling `.find_next_sibling()` on `None` would raise an error.

In [26]:
def get_row_value(box, label_text):
    label = box.find("div", string=label_text)
    if label is None:
        return None
    return label.find_next_sibling()


def get_row_text(box, label_text):
    value = get_row_value(box, label_text)
    if value is None:
        return None
    return value.get_text(" ", strip=True)

In [27]:
culture = get_row_text(box, "Culture")
medium = get_row_text(box, "Medium")
measurements = get_row_text(box, "Measurements")
location = get_row_text(box, "Location")

print(culture)
print(medium)
print(measurements)
print(location)

Netherlands
oil on fabric
Framed: 72.5 x 73.5 x 8.5 cm (28 9/16 x 28 15/16 x 3 3/8 in.); Unframed: 50.2 x 50.5 cm (19 3/4 x 19 7/8 in.)
222 Impressionism & Post-Impressionism


### The credit line and accession number

The `Credit Line` row's value is a `<div>` holding a link (the credit) *and* a `<span>` (the accession number):

In [28]:
credit = get_row_value(box, "Credit Line")

credit_line = credit.find("a").get_text(strip=True)
accession_number = credit.find("span").get_text(strip=True)

print(credit_line)
print(accession_number)

Bequest of Leonard C. Hanna Jr.
1958.31


### The rest of the relevant data

The page has a few more facts worth keeping: the artist's biography line, the license, the description, the
image address, and the page's own address.

In [29]:
# The artist's nationality and years: a <span> next to the artist's name
artist_block = box.find("h4").find_parent("div")
bio_text = artist_block.find_next_sibling("span").get_text(strip=True)
artist_bio = bio_text.strip("()")

print(artist_bio)

Dutch, 1853–1890


In [30]:
# The license: the words are in the <title> of the notice's <svg> icon
license_text = box.find("svg").find("title").get_text(strip=True)

print(license_text)

Public Domain


In [31]:
# The description: its label is an <h3> this time, and the value is the <span> after it
description_label = box.find("h3", string="Description")
description = description_label.find_next_sibling("span").get_text(" ", strip=True)

print(description[:80])

In May 1890, Vincent van Gogh arrived in Auvers, a small town north of Paris, wh


In [32]:
# The image address: in the <head>, in a <meta> tag's content attribute.
# We're saving the address only, not downloading the image.
image_url = soup.find("meta", property="og:image")["content"]

print(image_url[:60])

https://piction.clevelandart.org/cma/ump.di?e=DBA3E21E7303FB


### Putting it all in a dictionary

In [33]:
artwork = {
    "title": title,
    "artist": artist,
    "artist_bio": artist_bio,
    "date": date,
    "culture": culture,
    "medium": medium,
    "measurements": measurements,
    "credit_line": credit_line,
    "accession_number": accession_number,
    "location": location,
    "license": license_text,
    "description": description,
    "image_url": image_url,
    "url": url,
}

for key, value in artwork.items():
    print(key, "=", str(value)[:100])

title = Adeline Ravoux
artist = Vincent van Gogh
artist_bio = Dutch, 1853–1890
date = 1890
culture = Netherlands
medium = oil on fabric
measurements = Framed: 72.5 x 73.5 x 8.5 cm (28 9/16 x 28 15/16 x 3 3/8 in.); Unframed: 50.2 x 50.5 cm (19 3/4 x 19
credit_line = Bequest of Leonard C. Hanna Jr.
accession_number = 1958.31
location = 222 Impressionism & Post-Impressionism
license = Public Domain
description = In May 1890, Vincent van Gogh arrived in Auvers, a small town north of Paris, where he rented a room
image_url = https://piction.clevelandart.org/cma/ump.di?e=DBA3E21E7303FB54FA88AA7F4C77DAD98BD491FB080247C6178961
url = https://www.clevelandart.org/art/1958.31


Try asking for a label that isn't on the page, like `"Provenance"`, to see the `None` come back:

In [34]:
print(get_row_text(box, "Provenance"))

None


## 7. Scraping several artworks

To make a *dataset*, we repeat the steps for many works and collect the results:

1. Make a list of the pages' addresses.
2. Turn the single-page code into a function.
3. Loop over the list, scraping each page.
4. Collect the dictionaries in a list.

### Collecting the URLs

The collection search page lists an artist's works, but it builds its results with JavaScript, so `requests`
would get an empty page. A person using a browser sees the list just fine, so **this step is done by hand**:

1. Open an artwork page and click the artist's name, *Vincent van Gogh*. This opens the collection search,
   showing his works.
2. Open each work and copy its address. (It always ends with the accession number.)
3. Paste the addresses into a Python list.

At the time of writing, the collection has five works by Van Gogh. Paste each address into your browser to
check it before running anything: a typo means a `404` error.

In [35]:
urls = [
    "https://www.clevelandart.org/art/1958.31",
    "https://www.clevelandart.org/art/1947.209",
    "https://www.clevelandart.org/art/1958.32",
    "https://www.clevelandart.org/art/1958.30",
    "https://www.clevelandart.org/art/1948.303",
]

### Turning the scraper into a function

The function takes a page's HTML and its address, and **returns** the dictionary. Taking the HTML as an
argument means it doesn't care whether the HTML came from a live request or a saved file. It uses
`get_row_value()` and `get_row_text()` from above.

In [36]:
def scrape_artwork(html, url):
    soup = BeautifulSoup(html, "html.parser")
    box = soup.find(id="artwork-details-container")

    title = box.find("h3").get_text(strip=True)
    artist = box.find("h4").get_text(" ", strip=True)

    artist_block = box.find("h4").find_parent("div")
    bio_text = artist_block.find_next_sibling("span").get_text(strip=True)
    artist_bio = bio_text.strip("()")

    title_block = box.find("h3").find_parent("div")
    date = title_block.find_next_sibling("div").get_text(strip=True)

    culture = get_row_text(box, "Culture")
    medium = get_row_text(box, "Medium")
    measurements = get_row_text(box, "Measurements")
    location = get_row_text(box, "Location")

    credit = get_row_value(box, "Credit Line")
    credit_line = credit.find("a").get_text(strip=True)
    accession_number = credit.find("span").get_text(strip=True)

    license_text = box.find("svg").find("title").get_text(strip=True)

    description_label = box.find("h3", string="Description")
    description = description_label.find_next_sibling("span").get_text(" ", strip=True)

    image_url = soup.find("meta", property="og:image")["content"]

    return {
        "title": title,
        "artist": artist,
        "artist_bio": artist_bio,
        "date": date,
        "culture": culture,
        "medium": medium,
        "measurements": measurements,
        "credit_line": credit_line,
        "accession_number": accession_number,
        "location": location,
        "license": license_text,
        "description": description,
        "image_url": image_url,
        "url": url,
    }

### Looping over the URLs

For each address, the loop gets the page (from the saved copy if we have it, or from the museum if not),
calls the function, and **appends** the dictionary to a list. `fetch_page()` checks `response.ok` and pauses
10 seconds after each real request, so the first run takes about a minute. The page we already saved above
isn't fetched again.

**A note on running this in class:** if everyone runs this at the same moment, the museum's server gets all
those requests at once, far faster than the 10 seconds it asks for. In class we'll take turns, or work from
copies of the pages we're given. Put saved copies in the `cma_pages` folder, named like `cma_1958_31.html`, and
the loop uses them without making any requests.

In [37]:
artworks = []

for url in urls:
    accession = url.split("/")[-1]
    filename = "cma_pages/cma_" + accession.replace(".", "_") + ".html"

    html = fetch_page(url, filename)

    if html is not None:
        artwork = scrape_artwork(html, url)
        artworks.append(artwork)

When the loop finishes, `artworks` is a list of dictionaries, one per work, in the same shape as the `data` list from `csv.DictReader` in Part 7:

In [38]:
print(len(artworks))

for artwork in artworks:
    print(artwork["accession_number"], "|", artwork["title"], "|", artwork["date"], "|", artwork["medium"])

5
1958.31 | Adeline Ravoux | 1890 | oil on fabric
1947.209 | The Large Plane Trees (Road Menders at Saint-Rémy) | 1889 | oil on fabric
1958.32 | Two Poplars in the Alpilles near Saint-Rémy | 1889 | oil on fabric
1958.30 | Landscape with Wheelbarrow | 1883 | watercolor and opaque watercolor with black chalk on cream paper
1948.303 | Dr. Gachet | 1890 | Etching and drypoint, enhanced with black ink on laid paper


Everything from earlier parts works on it. The filter loop from Part 5 picks out the works made in oil:

In [39]:
oil_paintings = []

for artwork in artworks:
    if "oil" in artwork["medium"]:
        oil_paintings.append(artwork)

for artwork in oil_paintings:
    print(artwork["title"])

Adeline Ravoux
The Large Plane Trees (Road Menders at Saint-Rémy)
Two Poplars in the Alpilles near Saint-Rémy


The run also turned up differences between the works that our label-based code coped with:

* The drawing has an extra `Support` row. Because we look rows up by label, an extra row doesn't disturb anything.
* The location varies: `Not on view`, a gallery number, or a loan with dates. A column like this needs cleaning
  before it can be counted or mapped.
* The mediums vary widely.
* If the museum redesigns its pages, `scrape_artwork()` will most likely stop with an `AttributeError`. That's
  the signal to inspect the new page and update the code.

## 8. Saving the list as a CSV

`artworks` only exists while the notebook is running. To keep the dataset, we write it to a file. We'll start
with a **CSV**: the same process as "Saving a CSV file" in Part 9.

In [40]:
with open("van_gogh_artworks.csv", "w", encoding="utf-8") as file:
    writer = csv.DictWriter(file, artworks[0].keys())
    writer.writeheader()
    writer.writerows(artworks)

* `"w"` opens the file for writing, and replaces it if it exists.
* `artworks[0].keys()` gives the column names: the keys of the first dictionary, in order.
* `writeheader()` writes the column names as the first line, and `writerows()` writes one line per dictionary.

Every dictionary that `scrape_artwork()` returns has the same 14 keys, so they all fit the same columns. A
dictionary with an extra key would make `DictWriter` stop with a `ValueError`. Values that contain commas, like
`Dutch, 1853–1890`, are put in quotation marks by the writer.

**Check the file** by reading it back with `csv.DictReader`:

In [41]:
with open("van_gogh_artworks.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    loaded_csv = list(reader)

print(len(loaded_csv))
print(loaded_csv[0]["title"])
print(loaded_csv[4]["medium"])
print(loaded_csv == artworks)

5
Adeline Ravoux
Etching and drypoint, enhanced with black ink on laid paper
True


That comparison only holds because everything we scraped is already text. A CSV stores **everything as text**,
and has no way to write `None`: it's written as an empty cell and read back as `''`.

If accented letters look wrong when you open the file in Excel, try `encoding="utf-8-sig"` when writing it. If
you're on Windows and blank lines appear between the rows, add `newline=""` to `open()`.

## 9. Saving the list as JSON

**JSON** is another option, and a good fit for a list of dictionaries: it's exactly the shape JSON describes. It
also keeps things a CSV can't, like nested lists and dictionaries, and the difference between a missing value
(`null`) and an empty string.

In Part 9, `json.load()` read a JSON file. Its counterpart for writing is `json.dump()`:

In [42]:
with open("van_gogh_artworks.json", "w", encoding="utf-8") as file:
    json.dump(artworks, file, indent=4, ensure_ascii=False)

* `json.dump(data, file)` takes the data first and the file second. (`csv.DictWriter()` takes the file first.)
* `indent=4` spreads the output over many lines so a person can read it. Leave it out and the whole list is one
  long line.
* `ensure_ascii=False` keeps accented letters and dashes as they are. By default, `json` writes them as escape
  codes. Compare, using `json.dumps()` (with an *s*, which makes a *string* instead of writing a file):

In [43]:
title = artworks[1]["title"]

print(json.dumps(title))
print(json.dumps(title, ensure_ascii=False))

"The Large Plane Trees (Road Menders at Saint-R\u00e9my)"
"The Large Plane Trees (Road Menders at Saint-Rémy)"


Python's types become JSON's in a predictable way: a list becomes an array, a dictionary becomes an object,
strings stay strings, and `None` becomes `null`. If a work's page had no `Culture` row, the file would have
`"culture": null` for it.

**Check the file** by reading it back with `json.load()`:

In [44]:
with open("van_gogh_artworks.json", "r", encoding="utf-8") as file:
    loaded_json = json.load(file)

print(type(loaded_json))
print(len(loaded_json))
print(loaded_json[0]["title"])
print(loaded_json == artworks)

<class 'list'>
5
Adeline Ravoux
True


Open `van_gogh_artworks.json` in a text editor too: is it readable, and do the accents and dashes look right?

A word on what the files contain: text and addresses copied from the museum's pages. That's fine for class
work, but before you share or publish a dataset like this, check the museum's terms of use for what you may
reuse. The `license` field helps, but it's no substitute for reading the terms.

That's the whole pipeline: we checked that scraping was allowed, looked at how a page is built, pulled one artwork
out of its HTML tags, turned that into a function and ran it on a list of pages, and saved the list of
dictionaries as a CSV and as JSON.

## Your turn

**Challenge A: map the structure of a page.** Pick a site, check it, get its HTML once, find the container, and
walk through it with BeautifulSoup. Use the cells below for your work.
[Challenge A instructions](https://craigfahner.github.io/INFO-664/week7-challenge-a/)

### Challenge A: my site and checks

*(Site, address, `robots.txt` notes, terms of use notes, and how you got the HTML.)*

In [45]:
# your robots.txt check

In [46]:
# get your HTML once and save it

In [47]:
# find the container and walk through its children

### Challenge A: my map

| Data | Tag / label | How I would find it | Stable or fragile? |
| --- | --- | --- | --- |
|  |  |  |  |

**Challenge B: extract the data and save it as JSON.** Use your map to build a function that returns a
dictionary, collect at least three records, and save them.
[Challenge B instructions](https://craigfahner.github.io/INFO-664/week7-challenge-b/)

### Challenge B: my record

In [ ]:
# one fact at a time, printed as you go

In [ ]:
# your scrape function

In [ ]:
# your list of records

In [ ]:
# save as JSON, then read it back to check

### Challenge B: my data dictionary

| Key | What it means | Where it comes from on the page | Problems I noticed |
| --- | --- | --- | --- |
|  |  |  |  |